# Case Studies — Named Metro Deep Dives

Builds presentation-ready national trend charts and per-market profiles (Seattle, Philadelphia, Houston, Dallas, Boston, NYC, SF, San Jose) combining the structural gap trajectory, actual-vs-counterfactual space, top local SHAP drivers, and each market's own mean-reversion forecast into one multi-panel figure per metro.

**Requires:** run notebooks 01–05 first (uses their exported CSVs).

In [ ]:
"""
National Actual vs. Counterfactual — Clean Presentation Charts
================================================================
Standalone post-hoc script. Reads the two CSVs the main model script
already exports and regenerates the national actual-vs-counterfactual
trend chart in two versions:

  1. R&D_Available_SF_Total_Actual_vs_Counterfactual.png
     Unweighted, size-corrected. Uses Available_SF_Total and
     Counterfactual_Space_SF directly from the Results export.

  2. R&D_Available_SF_Total_AdvWeighted_Actual_vs_Counterfactual.png
     Advanced-industry-weighted, size-corrected. Uses
     Adv_Weighted_Available_SF_Total and Adv_Weighted_Counterfactual_SF
     from the ByMSA Adv-weighted export.

(Full original version history retained in git log / docs/methodology.md.)
"""

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams.update({"font.family": "serif", "font.size": 11})

BLUE = "#1a3a5c"
GRAY = "#888888"
RED = "#d62728"

RESULTS_CSV = "AvailSFTotal_Counterfactual_Results.csv"
BYMSA_ADV_CSV = "AvailSFTotal_AdvWeighted_Actual_vs_Counterfactual_ByMSA.csv"

NON_PRESENTATION_MSAS = ['Hattiesburg, MS', 'Gulfport-Biloxi, MS']

OUTPUT_UNWEIGHTED_PNG = "R&D_Available_SF_Total_Actual_vs_Counterfactual.png"
OUTPUT_WEIGHTED_PNG = "R&D_Available_SF_Total_AdvWeighted_Actual_vs_Counterfactual.png"


def make_chart(df, actual_col, counter_col, ylabel, title, subtitle, output_path):
    n_msas = df['MSA_Name'].nunique()
    if n_msas > 100:
        print(f"WARNING -- input to '{title}' has {n_msas} MSAs after filtering, not 100. "
              f"This looks like a v14 (102-MSA-trained) export, not v14b. The resulting "
              f"chart will show 100 MSAs' worth of points but will be built on values from "
              f"a model actually TRAINED on 102 MSAs -- not the same as a genuine v14b "
              f"(100-MSA-trained) run. Re-run against true v14b CSVs before trusting this figure.")

    yearly = df.groupby('Year').agg(
        Actual=(actual_col, 'mean'),
        Counter=(counter_col, 'mean'),
    ).reset_index()

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(yearly['Year'], yearly['Actual'] / 1e6, 'o-', color=BLUE, lw=2, ms=5,
            label='Actual Available SF Total')
    ax.plot(yearly['Year'], yearly['Counter'] / 1e6, 's--', color=GRAY, lw=1.5, ms=4,
            label='Counterfactual (pre-COVID structural model)')
    ax.fill_between(
        yearly['Year'], yearly['Actual'] / 1e6, yearly['Counter'] / 1e6,
        where=yearly['Year'] >= 2020, alpha=0.13, color=RED,
        label='COVID structural gap (2020-2023)')
    ax.axvspan(2019.5, 2023.5, alpha=0.06, color=RED)
    ax.axvline(2019.5, color=RED, lw=1.2, ls=':', alpha=0.5)

    ax.set_xlabel("Year", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(f"{title}\n{subtitle}", fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_xlim(2005.5, 2023.5)
    ax.grid(alpha=0.25, ls=':')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {output_path}")


# ══════════════════════════════════════════════════════════════════
# CHART 1 — Unweighted, size-corrected
# ══════════════════════════════════════════════════════════════════
results = pd.read_csv(RESULTS_CSV)
results_100 = results[~results['MSA_Name'].isin(NON_PRESENTATION_MSAS)]

make_chart(
    df=results_100,
    actual_col='Available_SF_Total',
    counter_col='Counterfactual_Space_SF',
    ylabel="Mean Available SF Total per MSA (Million SF)",
    title="R&D Available SF Total -- Actual vs. Counterfactual, 2006-2023",
    subtitle="100 MSAs | Size-corrected | Shaded = COVID structural gap (2020-2023)",
    output_path=OUTPUT_UNWEIGHTED_PNG,
)

# ══════════════════════════════════════════════════════════════════
# CHART 2 — Advanced-industry-weighted, size-corrected
# ══════════════════════════════════════════════════════════════════
adv = pd.read_csv(BYMSA_ADV_CSV)
adv = adv.rename(columns={
    'Adv_Weighted_Available_SF_Total': 'R&D_Weighted_Available_SF_Total',
    'Adv_Weighted_Counterfactual_SF': 'R&D_Weighted_Counterfactual_SF',
})
adv_100 = adv[~adv['MSA_Name'].isin(NON_PRESENTATION_MSAS)]

make_chart(
    df=adv_100,
    actual_col='R&D_Weighted_Available_SF_Total',
    counter_col='R&D_Weighted_Counterfactual_SF',
    ylabel="R&D-Weighted Mean Available SF Total per MSA (Million SF)",
    title="R&D Available SF Total -- R&D-Weighted Actual vs. Counterfactual, 2006-2023",
    subtitle="100 MSAs | Size-corrected, R&D-Weighted | Shaded = COVID structural gap (2020-2023)",
    output_path=OUTPUT_WEIGHTED_PNG,
)

print("\nDone -- both charts regenerated on the 100-MSA presentation set, version tag removed from titles.")

In [ ]:
"""
Market Case Studies — Seattle, Philadelphia, Houston, Dallas, Boston, NYC, SF/San Jose
=========================================================================
Pulls each target MSA's results across ALL FOUR methods from the
outputs the v14b pipeline already produces -- does NOT re-run or
re-derive anything. This is pure post-hoc assembly: for each target
metro, gather its row(s) from every CSV the main model, regional
script, and mean-reversion script already exported, and present them
together as a per-market profile.

TARGET MARKETS (exact MSA_Name strings as used throughout this
project -- VERIFY these match your data; the sanity check below will
fail loudly and list what it actually found if any don't match):
  - Seattle-Tacoma-Bellevue, WA           (priority: ***)
  - Philadelphia-Camden-Wilmington, PA-NJ-DE-MD  (priority: **)
  - Houston-Pasadena-The Woodlands, TX     (priority: ** -- name
    confirmed via the sanity check's keyword-match suggestion; the
    CBSA naming convention differs from the commonly-used
    "Houston-The Woodlands-Sugar Land" form)
  - Dallas-Fort Worth-Arlington, TX
  - Boston-Cambridge-Newton, MA-NH
  - New York-Newark-Jersey City, NY-NJ
  - San Francisco-Oakland-Fremont, CA
  - San Jose-Sunnyvale-Santa Clara, CA
  (San Francisco/San Jose are two separate MSAs in this project's
  panel, not one combined "Bay Area" unit -- both are pulled
  separately below.)

WHAT EACH MARKET'S PROFILE INCLUDES:
  METHOD 1 (Counterfactual): year-by-year Structural_Gap and
    Market_Category (2006-2023), 2020-2023 average unweighted and
    R&D-weighted gap, national rank (unweighted and R&D-weighted,
    plus Rank_Shift between them if this MSA appears in the rank-
    comparison export).
  METHOD 2 (Nascent Market): NOT APPLICABLE to these 7 markets by
    construction -- Method 2 in this project is restricted to the 20
    SMALLEST-base MSAs, and all 7 target markets here are large,
    established metros. This section reports Adv-industry LQ level
    and trend for context, without claiming Method 2's nascent-
    market classification applies.
  METHOD 3 (Regional): which Census region the MSA belongs to, and
    whether it appears in that region's Top-5 surplus/deficit list
    (unweighted or R&D-weighted).
  METHOD 4 (Mean-Reversion): Equilibrium, Deviation0, and convergence
    status (already converged vs. still reverting) as of 2023.

(Full original version history retained in git log / docs/methodology.md.)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams.update({"font.family": "serif", "font.size": 11})

BLUE = "#1a3a5c"
GRAY = "#c9c9c9"
RED = "#d62728"
GREEN = "#2ca02c"

RESULTS_CSV = "AvailSFTotal_Counterfactual_Results.csv"
BYMSA_ADV_CSV = "AvailSFTotal_AdvWeighted_Actual_vs_Counterfactual_ByMSA.csv"
COVID_AVG_CSV = "AvailSFTotal_COVID_Avg_Gap.csv"
COVID_ADVWEIGHTED_CSV = "AvailSFTotal_COVID_AdvWeighted_Gap.csv"
RANK_COMPARISON_CSV = "RankComparison_Unweighted_vs_RDWeighted.csv"  # optional
REGIONAL_TOP5_UNWEIGHTED_CSV = "Regional_Top5_Unweighted_v14b.csv"    # optional
REGIONAL_TOP5_RDWEIGHTED_CSV = "Regional_Top5_RDWeighted_v14b.csv"    # optional
MEANREVERSION_BYMSA_CSV = "MeanReversion_RDWeighted_ByMSA_v14b.csv"   # optional

PREDICT_YEARS = [2020, 2021, 2022, 2023]

TARGET_MSAS = [
    "Seattle-Tacoma-Bellevue, WA",
    "Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",
    "Houston-Pasadena-The Woodlands, TX",
    "Dallas-Fort Worth-Arlington, TX",
    "Boston-Cambridge-Newton, MA-NH",
    "New York-Newark-Jersey City, NY-NJ",
    "San Francisco-Oakland-Fremont, CA",
    "San Jose-Sunnyvale-Santa Clara, CA",
]

OUTPUT_PROFILE_CSV = "Market_CaseStudies_Combined_Profile.csv"
OUTPUT_TRAJECTORY_PNG = "market_casestudies_trajectories.png"

# ══════════════════════════════════════════════════════════════════
# 0. SANITY CHECK — confirm every target MSA name actually exists in
#    the data, and warn if any source file looks pre-v14b (>100 MSAs)
# ══════════════════════════════════════════════════════════════════
results = pd.read_csv(RESULTS_CSV)
_n_msas = results['MSA_Name'].nunique()
if _n_msas > 100:
    print(f"WARNING -- {RESULTS_CSV} contains {_n_msas} MSAs, not 100. This looks like "
          f"a pre-v14b export -- case-study numbers below will not reflect the 100-MSA-"
          f"trained model.")

missing_targets = [m for m in TARGET_MSAS if m not in results['MSA_Name'].unique()]
if missing_targets:
    print(f"\n{'!'*70}")
    print(f"WARNING -- {len(missing_targets)} target MSA name(s) NOT FOUND in {RESULTS_CSV}:")
    for m in missing_targets:
        print(f"    {m!r}")
    print(f"Check exact naming/punctuation against your actual data. Available MSA names")
    print(f"containing likely keywords, for reference:")
    for m in missing_targets:
        keyword = m.split('-')[0].split(',')[0]
        matches = [x for x in results['MSA_Name'].unique() if keyword.split()[0] in x]
        print(f"  Possible matches for {keyword!r}: {matches}")
    print(f"{'!'*70}")

found_targets = [m for m in TARGET_MSAS if m in results['MSA_Name'].unique()]
print(f"\n{len(found_targets)} of {len(TARGET_MSAS)} target MSAs confirmed present.")

# ══════════════════════════════════════════════════════════════════
# 1. LOAD ALL SOURCES (optional ones loaded defensively)
# ══════════════════════════════════════════════════════════════════
adv = pd.read_csv(BYMSA_ADV_CSV)
adv = adv.rename(columns={
    'Adv_Weighted_Available_SF_Total': 'R&D_Weighted_Available_SF_Total',
    'Adv_Weighted_Counterfactual_SF': 'R&D_Weighted_Counterfactual_SF',
})
covid_avg = pd.read_csv(COVID_AVG_CSV)
covid_adv = pd.read_csv(COVID_ADVWEIGHTED_CSV)

def try_load(path, label):
    try:
        return pd.read_csv(path)
    except FileNotFoundError:
        print(f"NOTE -- {path} not found; {label} will be omitted from case studies.")
        return None

rank_comp = try_load(RANK_COMPARISON_CSV, "rank-shift context")
regional_uw = try_load(REGIONAL_TOP5_UNWEIGHTED_CSV, "regional unweighted Top-5 context")
regional_rd = try_load(REGIONAL_TOP5_RDWEIGHTED_CSV, "regional R&D-weighted Top-5 context")
meanrev = try_load(MEANREVERSION_BYMSA_CSV, "Method 4 (mean-reversion) context")

covid_avg['Rank_Unweighted'] = covid_avg['Avg_Gap_2020_2023'].rank(ascending=False)
covid_adv['Rank_RDWeighted'] = covid_adv['Avg_Adv_Weighted_Gap_2020_2023'].rank(ascending=False)

# ══════════════════════════════════════════════════════════════════
# 2. PER-MARKET PROFILE ASSEMBLY
# ══════════════════════════════════════════════════════════════════
profile_rows = []

for msa in found_targets:
    print(f"\n{'='*70}")
    print(f"CASE STUDY — {msa}")
    print(f"{'='*70}")

    # --- METHOD 1: Counterfactual ---
    msa_full = results[results['MSA_Name'] == msa].sort_values('Year')
    msa_predict = msa_full[msa_full['Year'].isin(PREDICT_YEARS)]
    mean_gap = msa_predict['Structural_Gap'].mean()
    typical_category = msa_predict['Market_Category'].mode().iat[0] if len(msa_predict) else np.nan
    rank_uw_row = covid_avg[covid_avg['MSA_Name'] == msa]
    rank_uw = rank_uw_row['Rank_Unweighted'].iat[0] if len(rank_uw_row) else np.nan

    msa_adv_predict = adv[(adv['MSA_Name'] == msa) & (adv['Year'].isin(PREDICT_YEARS))]
    rank_rd_row = covid_adv[covid_adv['MSA_Name'] == msa]
    mean_rd_gap = rank_rd_row['Avg_Adv_Weighted_Gap_2020_2023'].iat[0] if len(rank_rd_row) else np.nan
    rank_rd = rank_rd_row['Rank_RDWeighted'].iat[0] if len(rank_rd_row) else np.nan

    rank_shift_note = "N/A (rank-comparison export not loaded)"
    if rank_comp is not None:
        rc_row = rank_comp[rank_comp['MSA_Name'] == msa]
        if len(rc_row):
            rank_shift_note = f"{rc_row['Rank_Shift'].iat[0]:+.0f} positions"

    print(f"\n  METHOD 1 — Counterfactual:")
    if pd.notna(rank_uw):
        print(f"    2020-2023 avg Structural_Gap (unweighted): {mean_gap:+.4f} "
              f"(national rank {rank_uw:.0f} of 100)")
    else:
        print(f"    2020-2023 avg Structural_Gap (unweighted): {mean_gap:+.4f}")
    print(f"    Typical Market_Category: {typical_category}")
    if pd.notna(rank_rd):
        print(f"    2020-2023 avg R&D-weighted gap: {mean_rd_gap:+,.0f} SF "
              f"(national rank {rank_rd:.0f} of 100)")
    else:
        print(f"    2020-2023 avg R&D-weighted gap: {mean_rd_gap}")
    print(f"    Rank shift (unweighted -> R&D-weighted): {rank_shift_note}")

    # --- METHOD 2: Nascent Market (context only, not applicable) ---
    lq_recent = msa_full[msa_full['Year'].isin([2021, 2022, 2023])]['LQ_AdvInd_Emp'].mean()
    lq_anchor = msa_full[msa_full['Year'].isin([2015, 2016, 2017, 2018])]['LQ_AdvInd_Emp'].mean()
    print(f"\n  METHOD 2 — Nascent Market: NOT APPLICABLE (restricted to the 20 "
          f"smallest-base MSAs; {msa} is a large, established market)")
    print(f"    For context only -- Adv-industry LQ, 2015-18 anchor: {lq_anchor:.3f}, "
          f"2021-23 recent: {lq_recent:.3f} ({'rising' if lq_recent > lq_anchor else 'falling'} "
          f"concentration)")

    # --- METHOD 3: Regional ---
    region_note = "N/A (regional Top-5 exports not loaded)"
    if regional_uw is not None:
        in_uw_top5 = regional_uw[regional_uw['MSA_Name'] == msa]
        if len(in_uw_top5):
            row = in_uw_top5.iloc[0]
            region_note = f"{row['Region']} region, {row['Rank_Type']} (unweighted)"
        else:
            region_note = "does not appear in any region's unweighted Top-5"
    print(f"\n  METHOD 3 — Regional: {region_note}")
    if regional_rd is not None:
        in_rd_top5 = regional_rd[regional_rd['MSA_Name'] == msa]
        if len(in_rd_top5):
            row = in_rd_top5.iloc[0]
            print(f"    Also appears in R&D-weighted regional Top-5: {row['Rank_Type']} "
                  f"({row['R&D_Weighted_Gap_SF']:+,.0f} SF)")

    # --- METHOD 4: Mean-Reversion ---
    equilibrium, deviation0, converged = np.nan, np.nan, np.nan
    if meanrev is not None:
        mr_row = meanrev[meanrev['MSA_Name'] == msa]
        if len(mr_row):
            # NOTE: Equilibrium_ForForecast is not exported directly by
            # meanreversion_rdweighted_v14b.py (a bug in that script's
            # export column list -- it computes this value but never
            # writes it out). Derived here instead from the two
            # components that ARE exported (Residual_Equilibrium +
            # Bias_2023), which reproduces it exactly since that's how
            # the mean-reversion script computes it internally.
            if 'Equilibrium_ForForecast' in mr_row.columns:
                equilibrium = mr_row['Equilibrium_ForForecast'].iat[0]
            elif 'Residual_Equilibrium' in mr_row.columns and 'Bias_2023' in mr_row.columns:
                equilibrium = mr_row['Residual_Equilibrium'].iat[0] + mr_row['Bias_2023'].iat[0]
            deviation0 = mr_row['Deviation0'].iat[0]
            converged = mr_row['Already_Converged_2023'].iat[0]
    print(f"\n  METHOD 4 — Mean-Reversion:")
    if pd.notna(equilibrium):
        print(f"    Equilibrium (long-run target): {equilibrium:+.4f}")
    else:
        print("    Equilibrium: N/A (mean-reversion export not loaded)")
    if pd.notna(deviation0):
        print(f"    Deviation0 (2023): {deviation0:+.4f}")
    print(f"    Convergence status: "
          f"{'Already converged' if converged is True else 'Still reverting' if converged is False else 'N/A'}")

    profile_rows.append({
        'MSA_Name': msa,
        'Typical_Market_Category': typical_category,
        'Mean_Gap_Unweighted_2020_2023': mean_gap,
        'Rank_Unweighted': rank_uw,
        'Mean_Gap_RDWeighted_2020_2023': mean_rd_gap,
        'Rank_RDWeighted': rank_rd,
        'LQ_AdvInd_Emp_Anchor_2015_2018': lq_anchor,
        'LQ_AdvInd_Emp_Recent_2021_2023': lq_recent,
        'Equilibrium_ForForecast': equilibrium,
        'Deviation0_2023': deviation0,
        'Already_Converged_2023': converged,
    })

profile_df = pd.DataFrame(profile_rows)
profile_df.to_csv(OUTPUT_PROFILE_CSV, index=False)
print(f"\n{'='*70}\nSaved combined profile: {OUTPUT_PROFILE_CSV}\n{'='*70}")
print(profile_df.round(4).to_string(index=False))

# ══════════════════════════════════════════════════════════════════
# 3. SMALL-MULTIPLE TRAJECTORY FIGURE — Structural_Gap over time,
#    2006-2023, one panel per target MSA
# ══════════════════════════════════════════════════════════════════
n = len(found_targets)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4.2 * nrows))
axes = np.array(axes).flatten()

for i, msa in enumerate(found_targets):
    ax = axes[i]
    sub = results[results['MSA_Name'] == msa].sort_values('Year')
    ax.plot(sub['Year'], sub['Structural_Gap'], 'o-', color=BLUE, lw=2, ms=4)
    ax.axhline(0, color='black', lw=0.8, ls=':')
    ax.axvspan(2019.5, 2023.5, alpha=0.08, color=RED)
    ax.set_title(msa.split(',')[0], fontsize=11, fontweight='bold')
    ax.set_xlabel("Year", fontsize=9)
    ax.set_ylabel("Structural Gap", fontsize=9)
    ax.grid(alpha=0.25, ls=':')

for j in range(n, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    "Structural Gap Trajectories, 2006-2023 — Market Case Studies [SIZE-CORRECTED, 100-MSA PANEL]",
    fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_TRAJECTORY_PNG, dpi=300, bbox_inches='tight')
plt.show()
print(f"\nSaved: {OUTPUT_TRAJECTORY_PNG}")

print(f"\n{'='*60}\nDONE — Market case studies complete\n{'='*60}")

In [ ]:
"""
Section 7f (NEW) — Individual MSA Case-Study Dashboards
=========================================================================
Produces ONE multi-panel figure per case-study market, combining
everything the mini case-study write-ups actually draw on: the
Structural_Gap trajectory, actual-vs-counterfactual available space,
the top local SHAP drivers for that MSA's 2023 prediction, and that
specific MSA's own mean-reversion forecast path (not the national
aggregate -- each MSA gets its own equilibrium/deviation/lambda
trajectory, computed from its own Equilibrium_ForForecast and
Deviation0).

DEFAULT TARGET SET: Seattle, Houston, Philadelphia -- the three
markets covered in the mini case studies. Easily extended to all 8
case-study markets by uncommenting the additional names below.

FOUR PANELS PER MARKET:
  A. Structural_Gap trajectory, 2006-2023 (size-corrected, unweighted)
  B. Actual vs. Counterfactual Available_SF_Total, 2006-2023
  C. Top 8 SHAP drivers of this MSA's 2023 prediction (from Section 7e)
  D. This MSA's own mean-reversion forecast to 2045 (median path only,
     no Monte Carlo band -- a per-MSA point forecast, not the national
     weighted aggregate reported elsewhere in this paper)

(Full original version history retained in git log / docs/methodology.md.)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams.update({"font.family": "serif", "font.size": 10})

BLUE = "#1a3a5c"
GRAY = "#888888"
RED = "#d62728"
TEAL = "#17becf"

RESULTS_CSV = "AvailSFTotal_Counterfactual_Results.csv"
SHAP_WIDE_CSV = "SHAP_Local_Explanations_CaseStudyMSAs_2023_wide.csv"
MEANREVERSION_BYMSA_CSV = "MeanReversion_RDWeighted_ByMSA_v14b.csv"

LAST_ACTUAL_YEAR = 2023
FORECAST_YEARS = list(range(2023, 2046))
LAMBDA_MEAN = 0.30  # matches the assumed reversion speed used throughout this project

# Default: the three markets covered in the mini case studies. Add
# any of the other 5 case-study markets here if you want dashboards
# for them too -- no other code changes needed.
TARGET_MSAS = [
    "Seattle-Tacoma-Bellevue, WA",
    "Houston-Pasadena-The Woodlands, TX",
    "Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",
    # "Dallas-Fort Worth-Arlington, TX",
    # "Boston-Cambridge-Newton, MA-NH",
    # "New York-Newark-Jersey City, NY-NJ",
    # "San Francisco-Oakland-Fremont, CA",
    # "San Jose-Sunnyvale-Santa Clara, CA",
]

print(f"\n{'='*70}")
print("SECTION 7f — INDIVIDUAL MSA CASE-STUDY DASHBOARDS")
print(f"{'='*70}")

# ══════════════════════════════════════════════════════════════════
# 0. LOAD SOURCES
# ══════════════════════════════════════════════════════════════════
results = pd.read_csv(RESULTS_CSV)

try:
    shap_wide = pd.read_csv(SHAP_WIDE_CSV, index_col=0)
except FileNotFoundError:
    shap_wide = None
    print(f"NOTE -- {SHAP_WIDE_CSV} not found; Panel C (SHAP drivers) will be omitted. "
          f"Run shap_individual_msa_importance.py first to enable it.")

try:
    meanrev = pd.read_csv(MEANREVERSION_BYMSA_CSV)
    if 'Equilibrium_ForForecast' not in meanrev.columns and \
       {'Residual_Equilibrium', 'Bias_2023'}.issubset(meanrev.columns):
        meanrev['Equilibrium_ForForecast'] = meanrev['Residual_Equilibrium'] + meanrev['Bias_2023']
except FileNotFoundError:
    meanrev = None
    print(f"NOTE -- {MEANREVERSION_BYMSA_CSV} not found; Panel D (forecast) will be omitted.")

missing = [m for m in TARGET_MSAS if m not in results['MSA_Name'].unique()]
if missing:
    print(f"\nWARNING -- target MSA(s) not found in {RESULTS_CSV}: {missing}")
found_targets = [m for m in TARGET_MSAS if m in results['MSA_Name'].unique()]
print(f"\n{len(found_targets)} of {len(TARGET_MSAS)} target MSAs confirmed present.")

# ══════════════════════════════════════════════════════════════════
# 1. BUILD ONE DASHBOARD PER MSA
# ══════════════════════════════════════════════════════════════════
for msa in found_targets:
    print(f"\nBuilding dashboard for {msa}...")
    msa_data = results[results['MSA_Name'] == msa].sort_values('Year')

    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    fig.suptitle(f"Case-Study Dashboard — {msa}", fontsize=15, fontweight='bold')

    # --- Panel A: Structural_Gap trajectory ---
    ax = axes[0, 0]
    ax.plot(msa_data['Year'], msa_data['Structural_Gap'], 'o-', color=BLUE, lw=2, ms=5)
    ax.axhline(0, color='black', lw=0.8, ls=':')
    ax.axvspan(2019.5, 2023.5, alpha=0.08, color=RED)
    ax.set_xlabel("Year")
    ax.set_ylabel("Structural Gap (log units)")
    ax.set_title("A. Structural Gap Trajectory, 2006-2023", fontsize=11, fontweight='bold')
    ax.grid(alpha=0.25, ls=':')

    # --- Panel B: Actual vs. Counterfactual Available SF ---
    ax = axes[0, 1]
    ax.plot(msa_data['Year'], msa_data['Available_SF_Total'] / 1e6, 'o-',
             color=BLUE, lw=2, ms=5, label='Actual')
    ax.plot(msa_data['Year'], msa_data['Counterfactual_Space_SF'] / 1e6, 's--',
             color=GRAY, lw=1.5, ms=4, label='Counterfactual')
    ax.fill_between(msa_data['Year'], msa_data['Available_SF_Total'] / 1e6,
                     msa_data['Counterfactual_Space_SF'] / 1e6,
                     where=msa_data['Year'] >= 2020, alpha=0.13, color=RED)
    ax.axvspan(2019.5, 2023.5, alpha=0.05, color=RED)
    ax.set_xlabel("Year")
    ax.set_ylabel("Available SF Total (Million SF)")
    ax.set_title("B. Actual vs. Counterfactual Space, 2006-2023", fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25, ls=':')

    # --- Panel C: Top SHAP drivers, 2023 ---
    ax = axes[1, 0]
    if shap_wide is not None and msa in shap_wide.index:
        msa_shap = shap_wide.loc[msa].sort_values(key=np.abs, ascending=False).head(8)
        colors = [TEAL if v > 0 else RED for v in msa_shap.values]
        ax.barh(msa_shap.index[::-1], msa_shap.values[::-1], color=colors[::-1], alpha=0.85)
        ax.axvline(0, color='black', lw=1, ls='--')
        ax.set_xlabel("SHAP Value (2023 prediction)")
        ax.set_title("C. Top SHAP Drivers, 2023", fontsize=11, fontweight='bold')
        ax.tick_params(axis='y', labelsize=8)
        ax.grid(alpha=0.25, ls=':', axis='x')
    else:
        ax.text(0.5, 0.5, "SHAP data not available\n(run shap_individual_msa_importance.py)",
                ha='center', va='center', fontsize=10, color=GRAY, transform=ax.transAxes)
        ax.set_title("C. Top SHAP Drivers, 2023", fontsize=11, fontweight='bold')
        ax.axis('off')

    # --- Panel D: MSA-specific mean-reversion forecast ---
    ax = axes[1, 1]
    if meanrev is not None and msa in meanrev['MSA_Name'].values:
        mr_row = meanrev[meanrev['MSA_Name'] == msa].iloc[0]
        equilibrium = mr_row['Equilibrium_ForForecast']
        deviation0 = mr_row['Deviation0']
        gap_2023 = mr_row['Gap_2023_corrected']

        forecast_path = [equilibrium + deviation0 * (1 - LAMBDA_MEAN) ** (y - LAST_ACTUAL_YEAR)
                          for y in FORECAST_YEARS]
        ax.plot(FORECAST_YEARS, forecast_path, '-', color=RED, lw=2, label='Forecast (median \u03bb=0.30)')
        ax.scatter([LAST_ACTUAL_YEAR], [gap_2023], color=BLUE, s=60, zorder=5, label='2023 actual')
        ax.axhline(equilibrium, color=TEAL, lw=1.2, ls=':',
                   label=f'Long-run target ({equilibrium:+.3f})')
        ax.set_xlabel("Year")
        ax.set_ylabel("Structural Gap (log units)")
        ax.set_title("D. MSA-Specific Mean-Reversion Forecast to 2045", fontsize=11, fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.25, ls=':')
        converged = mr_row.get('Already_Converged_2023', None)
        status = "Already converged" if converged is True else "Still reverting" if converged is False else ""
        if status:
            ax.text(0.02, 0.02, status, transform=ax.transAxes, fontsize=8,
                    color=GRAY, style='italic', va='bottom')
    else:
        ax.text(0.5, 0.5, "Mean-reversion data not available\n(run meanreversion_rdweighted_v14b.py)",
                ha='center', va='center', fontsize=10, color=GRAY, transform=ax.transAxes)
        ax.set_title("D. MSA-Specific Mean-Reversion Forecast to 2045", fontsize=11, fontweight='bold')
        ax.axis('off')

    plt.tight_layout()
    safe_name = msa.replace(',', '').replace(' ', '_').replace('-', '_')
    output_png = f"casestudy_dashboard_{safe_name}.png"
    plt.savefig(output_png, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"  Saved: {output_png}")

print(f"\n{'='*60}\nDONE — Section 7f complete\n{'='*60}")